# 02 · A LoRA adapter, in real numbers

What GRPO actually changed in one specialist. Set `TASK` below.

Each specialist folder has:

| Path | What it holds |
|---|---|
| `adapter/adapter_config.json` | LoRA settings: rank `r`, `lora_alpha`, target modules. |
| `adapter/adapter_model.safetensors` | The trained numbers: two small matrices `A`, `B` per target. |
| `merged/model.safetensors` | Full model = base + LoRA update, saved in bf16. This is what eval and merging use. |
| `run_info.json` | Config, git commit, training log. |

In [ ]:
import sys, json, struct
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, torch, matplotlib.pyplot as plt
from safetensors import safe_open
from src.merge.merge import LazyWeights

BASE = "Qwen/Qwen2.5-0.5B-Instruct"     # HF id (read from your HF cache) or a local folder
SEED = 42
TASKS = ["gsm8k", "logic", "code"]
METHODS = ["avg", "task_arithmetic", "ties"]
MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SPEC = {t: ROOT / f"checkpoints/specialists/{t}/seed{SEED}" for t in TASKS}
MERGED = {m: ROOT / f"checkpoints/merged/{m}" for m in METHODS}

# fixed colours: tasks always blue/orange/aqua, methods always yellow/magenta/green
C_TASK = dict(zip(TASKS, ["#2a78d6", "#eb6834", "#1baf7a"]))
C_METHOD = dict(zip(METHODS, ["#eda100", "#e87ba4", "#008300"]))
C_MOD = dict(zip(MODULES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]))

torch.set_printoptions(precision=4, sci_mode=False, linewidth=150)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e4e0", "axes.axisbelow": True})

def sub(module):
    return "self_attn" if module in MODULES[:4] else "mlp"

def wname(layer, module):
    return f"model.layers.{layer}.{sub(module)}.{module}.weight"

base = LazyWeights(BASE)
cfg = json.loads((base.path / "config.json").read_text())
N_LAYERS = cfg["num_hidden_layers"]
print("base folder:", base.path)

In [ ]:
TASK = "logic"
spec_dir = SPEC[TASK]
acfg = json.loads((spec_dir / "adapter" / "adapter_config.json").read_text())
r, alpha = acfg["r"], acfg["lora_alpha"]
scale = alpha / r
for k in ["r", "lora_alpha", "lora_dropout", "target_modules", "base_model_name_or_path"]:
    print(f"{k:<24} {acfg.get(k)}")
print(f"{'scale = alpha / r':<24} {scale}")

## 1. What LoRA stores

A full update to a matrix `W` [out × in] would need out × in numbers.
LoRA learns two thin matrices instead:

- `A`: [r × in], starts random
- `B`: [out × r], starts at **zero**

The update is **ΔW = (alpha / r) · B · A**. It starts at exactly 0 and has rank ≤ r = 16.
Trained weight = W + ΔW. Base W never changes during training.

In [ ]:
ad = safe_open(spec_dir / "adapter" / "adapter_model.safetensors", "pt")
akeys = list(ad.keys())
print(len(akeys), "tensors in the adapter (2 per target matrix)\n")
for k in akeys[:4]:
    print(f"{k:<70} {tuple(ad.get_slice(k).get_shape())}")

def lora(layer, module):
    p = f"base_model.model.model.layers.{layer}.{sub(module)}.{module}"
    return ad.get_tensor(p + ".lora_A.weight").float(), ad.get_tensor(p + ".lora_B.weight").float()

n_lora = sum(int(np.prod(ad.get_slice(k).get_shape())) for k in akeys)
n_full = sum(int(np.prod(base.get(wname(l, m)).shape)) for l in range(N_LAYERS) for m in MODULES)
print(f"\nLoRA numbers: {n_lora / 1e6:.2f}M  vs  {n_full / 1e6:.1f}M in the matrices they modify ({n_lora / n_full:.1%})")

## 2. A, B and ΔW for one matrix

In [ ]:
L, M = N_LAYERS // 2, "q_proj"
A, B = lora(L, M)
print(f"A {tuple(A.shape)}  (first 4 rows, 8 cols)\n{A[:4, :8]}\n")
print(f"B {tuple(B.shape)}  (first 8 rows, 4 cols)\n{B[:8, :4]}")

In [ ]:
W = base.get(wname(L, M)).float()
dW = scale * B @ A
print(f"ΔW = {scale} · B @ A   {tuple(dW.shape)}   corner:\n{dW[:5, :6]}\n")
print(f"base W corner:\n{W[:5, :6]}\n")
print(f"size of update: |ΔW| / |W| = {(dW.norm() / W.norm()).item():.2e}")
print(f"typical entry:  |W| median {W.abs().median():.2e}   |ΔW| median {dW.abs().median():.2e}")

## 3. ΔW is low-rank

Singular values say how many independent directions a matrix uses.
Base `W` uses all of them. `ΔW` has at most 16 non-zero ones: the LoRA rank.

In [ ]:
sW, sD = torch.linalg.svdvals(W), torch.linalg.svdvals(dW)
print("non-zero singular values of ΔW:", int((sD > sD[0] * 1e-4).sum()), " (rank r =", r, ")")
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.semilogy(sW / sW[0], color="#2a78d6", lw=2, label="base W")
ax.semilogy((sD / sD[0]).clamp(min=1e-8), color="#eb6834", lw=2, label="ΔW (LoRA)")
ax.axvline(r - 1, color="#6b6b6b", ls="--", lw=1); ax.text(r, 0.5, f" r = {r}", color="#6b6b6b")
ax.set_xlabel("index"); ax.set_ylabel("singular value / largest"); ax.set_ylim(1e-8, 2)
ax.legend(frameon=False); ax.set_title(f"layer {L} {M}")
plt.tight_layout(); plt.show()

## 4. Does `merged/` equal base + ΔW?

`merged/` was saved in bf16. Section 4 of notebook 01 showed that bf16 rounds away changes
smaller than roughly 0.4% of a weight. So some of ΔW should be lost.

- **saved ΔW** = merged − base, read from disk
- **exact ΔW** = scale · B · A, from the adapter

In [ ]:
spec = LazyWeights(spec_dir / "merged")
Ws = spec.get(wname(L, M)).float()
saved = Ws - W
same = (Ws == W).float().mean().item()
rel = (dW.abs() / W.abs().clamp(min=1e-12)).flatten()
print(f"entries identical to base after saving : {same:.1%}")
print(f"entries with |ΔW| < 0.39% of |W|       : {(rel < 2 ** -8).float().mean():.1%}  (expected to round away)")
print(f"|saved ΔW| / |exact ΔW|                : {(saved.norm() / dW.norm()).item():.3f}")
cos = torch.nn.functional.cosine_similarity(saved.flatten(), dW.flatten(), dim=0).item()
print(f"cosine(saved ΔW, exact ΔW)             : {cos:.3f}")

idx = torch.randperm(dW.numel())[:20000]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(dW.flatten()[idx], saved.flatten()[idx], s=2, color="#2a78d6", alpha=0.4)
lim = dW.abs().max().item()
axes[0].plot([-lim, lim], [-lim, lim], color="#6b6b6b", lw=1, ls="--")
axes[0].set_xlabel("exact ΔW (adapter)"); axes[0].set_ylabel("saved ΔW (merged − base)")
axes[0].set_title("each dot = one weight")
axes[1].hist(np.log10(rel.clamp(min=1e-8).numpy()), bins=120, color="#2a78d6")
axes[1].axvline(np.log10(2 ** -8), color="#eb6834", lw=2)
axes[1].text(np.log10(2 ** -8), axes[1].get_ylim()[1] * 0.9, " bf16 rounding limit", color="#23373b", fontsize=9)
axes[1].set_xlabel("log10( |ΔW| / |W| ) per entry"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()

Reading it: dots on the dashed line survived saving. Dots on the horizontal 0 line were rounded away.
Rounding is not random noise: a lost entry becomes 0, a kept one becomes a step of bf16.

`merge.py` builds task vectors from `merged/`, so it merges the **saved** ΔW, not the exact one.
If a large share is lost, merging from adapters in float32 is an option. Check section 6 across all layers first.

## 5. One matrix as a picture: W vs ΔW

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (mat, title) in zip(axes, [(W, "base W"), (dW, "exact ΔW"), (saved, "saved ΔW")]):
    patch = mat[:64, :64]
    v = max(patch.abs().quantile(0.99).item(), 1e-12)
    im = ax.imshow(patch, cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(f"{title} (scale ±{v:.1e})", fontsize=10); ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(f"layer {L} {M}, corner 64×64"); plt.tight_layout(); plt.show()

## 6. Where each specialist changed the model

`|ΔW| / |W|` for all 168 target matrices (exact ΔW from the adapters), for every specialist.
Same colour scale in all three, so they are comparable. Also: share of entries lost to bf16 per specialist.

In [ ]:
wnorm = {(l, m): base.get(wname(l, m)).float().norm().item() for l in range(N_LAYERS) for m in MODULES}

def change_map(task):
    h = safe_open(SPEC[task] / "adapter" / "adapter_model.safetensors", "pt")
    s = json.loads((SPEC[task] / "adapter" / "adapter_config.json").read_text())
    s = s["lora_alpha"] / s["r"]
    out = np.zeros((N_LAYERS, len(MODULES)))
    for l in range(N_LAYERS):
        for j, m in enumerate(MODULES):
            p = f"base_model.model.model.layers.{l}.{sub(m)}.{m}"
            d = s * h.get_tensor(p + ".lora_B.weight").float() @ h.get_tensor(p + ".lora_A.weight").float()
            out[l, j] = d.norm().item() / wnorm[(l, m)]
    return out

maps = {t: change_map(t) for t in TASKS if (SPEC[t] / "adapter").exists()}
vmax = max(m.max() for m in maps.values())
fig, axes = plt.subplots(1, len(maps), figsize=(4.2 * len(maps), 5.5), sharey=True)
for ax, (t, mp) in zip(np.atleast_1d(axes), maps.items()):
    im = ax.imshow(mp, cmap="Blues", vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(MODULES)), [m.replace("_proj", "") for m in MODULES], rotation=45)
    ax.set_title(f"{t}   mean {mp.mean():.1e}"); ax.grid(False)
    ax.set_yticks(range(0, N_LAYERS, max(1, N_LAYERS // 6)))
np.atleast_1d(axes)[0].set_ylabel("layer")
fig.colorbar(im, ax=axes, shrink=0.7, label="|ΔW| / |W|")
plt.show()

In [ ]:
lost = {}
for t in TASKS:
    s = LazyWeights(SPEC[t] / "merged")
    same = tot = 0
    for l in range(N_LAYERS):
        for m in MODULES:
            a, b = s.get(wname(l, m)), base.get(wname(l, m))
            same += (a == b).sum().item(); tot += a.numel()
    lost[t] = same / tot
    print(f"{t:<6} entries still identical to base after saving: {lost[t]:.1%}")

## 7. Size of ΔW entries vs W entries

Both on a log scale. The gap between the two peaks is how small RL updates are compared with the weights.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(np.log10(W.abs().flatten().clamp(min=1e-10).numpy()), bins=150, color="#2a78d6", alpha=0.85, label="|W| base")
ax.hist(np.log10(dW.abs().flatten().clamp(min=1e-10).numpy()), bins=150, color="#eb6834", alpha=0.85, label="|ΔW| LoRA")
ax.set_xlabel("log10 |value|"); ax.set_ylabel("count"); ax.legend(frameon=False)
ax.set_title(f"layer {L} {M}, {TASK} specialist")
plt.tight_layout(); plt.show()